In [1]:
!pip install -q transformers accelerate bitsandbytes gradio pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


In [2]:

import gradio as gr
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig
)

In [4]:
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

print("GPU available:", torch.cuda.is_available())

GPU available: False


In [5]:
MODEL = "microsoft/Phi-3-mini-4k-instruct" #you can change the model to any other model


In [6]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    quantization_config=quant_config
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [8]:
def prompt_builder(theme,language):

    system_prompt = f"""
You are a movie title expert.
You generate movie titles based on the given theme and language.
Generate realistic movie titles only.
The movie titles should be in {language}.
The movie should be short and catchy.
Generate upto 10 movie titles.
No explanations.
"""

    example = """
Example 1: Theme = animal
The movie title generated should be like
Anaconda
Snakes on a Plane
Jaws
Hotel for Dogs
Beverly Hills Chihuahua
Good Boy!
101 Dalmatians
Never Cry Wolf
The Doberman Gang
Dolphin Tale

Example 2: Theme = history
The movie title generated should be like
Saving Private Ryan
Gladiator
Schindler's List
Blood Diamond
Black Hawk Down
The Last of the Mohicans
Braveheart
Apollo 13
"""

    user = f"""
Suggest movie titles for the theme: {theme} and language: {language}
"""

    return system_prompt + example + user

In [17]:
def generate_names(theme,language):
    prompt = prompt_builder(theme,language)

    if(torch.cuda.is_available()):
      inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    else:
      inputs = tokenizer(prompt, return_tensors="pt")

    output = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = output[0][inputs.input_ids.shape[-1]:]

    text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

        # --- Extract possible names ---
    lines = text.split("\n")
    names = []
    for line in lines:
        if any(ch.isalpha() for ch in line):
            clean = line.strip()
            if "." in clean:
                clean = clean.split(".", 1)[1].strip()
            if len(clean.split()) <= 3 and not clean.lower().startswith("generate"):
                names.append(clean)

    # remove duplicates and limit
    names = list(dict.fromkeys(names))[:10]

    if not names:
        names = ["Model didn't generate recognizable names. Try again."]

    return "\n".join(names)

In [18]:
def run_app():
    with gr.Blocks(gr.themes.Ocean() , title="Movie name generator") as demo:
        # gr.Markdown("# Movie name generator")
        gr.Markdown("# Generates synthetic Movie names based on provided theme using Hugging Face Transformers with manual tokenization and decoding.")

        language = gr.Dropdown(
            ["English", "Hindi", "Spanish", "French"],
            label="Select Language",
            value="English"
        )
        theme = gr.Textbox(label="Enter Theme", lines=1)
        output = gr.Textbox(label="Generated Movie Names", lines=10)
        generate_btn = gr.Button("Generate Names")

        generate_btn.click(fn=generate_names, inputs=[theme,language], outputs=output)
    demo.launch(share=False)

In [19]:
run_app()

/tmp/ipykernel_1378/183974905.py:2: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(gr.themes.Ocean() , title="Movie name generator") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>